# DTDP - Dynamic Train Dispatching Problem

**A minimal NP-hard problem for Japan railway scheduling**

This notebook walks through:
1. Problem structure (Graph G = (V, E))
2. Generating instances
3. Evaluating solutions
4. Visualizing results
5. ShinkaEvolve integration

---
## 1. The Problem

```
Given:  Railway network G = (V, E) + trains
Find:   Departure time for each train
Goal:   Minimize conflicts + delays
```

**Why NP-hard?** Single-track segments mean trains conflict. Deciding who waits where creates exponential possibilities.

In [1]:
# Import modules
from problem import generate, Instance, RailGraph, Train
from fitness import evaluate, Schedule
from solver import naive, greedy_fcfs, greedy_priority, direction_wave, benchmark, print_benchmark
from visualize import show_graph, show_trains, show_schedule, show_timespace, report
from shinka import DTDPProblem

---
## 2. Graph Structure: G = (V, E)

The railway network is a weighted graph:
- **V** = stations (nodes)
- **E** = track segments with travel times (edges)

In [2]:
# Generate a simple instance
inst = generate(num_stations=5, num_trains=6, seed=42)

print(show_graph(inst.graph))

Graph: |V|=5, |E|=4

[0]--2--[1]--2--[2]--4--[3]--3--[4]


In [3]:
# Graph internals
print(f"|V| = {len(inst.graph.vertices)} stations")
print(f"|E| = {len(inst.graph.edges)} directed edges")
print()
print("Edges (bidirectional):")
seen = set()
for (u, v), t in inst.graph.edges.items():
    edge = tuple(sorted([u, v]))
    if edge not in seen:
        seen.add(edge)
        print(f"  {u} ↔ {v} : {t} time units")

|V| = 5 stations
|E| = 8 directed edges

Edges (bidirectional):
  0 ↔ 1 : 2 time units
  1 ↔ 2 : 2 time units
  2 ↔ 3 : 4 time units
  3 ↔ 4 : 3 time units


In [4]:
# Graphviz DOT format (for rendering)
print(inst.graph.to_dot())

graph Rail {
  rankdir=LR;
  node [shape=circle, style=filled, fillcolor=lightblue];
  0 -- 1 [label="2"];
  1 -- 2 [label="2"];
  2 -- 3 [label="4"];
  3 -- 4 [label="3"];
}


---
## 3. Trains

Each train has:
- `origin` → `destination` (path computed via BFS)
- `earliest` departure time
- `priority` (higher = more important)

In [5]:
print(show_trains(inst))

Trains: 6

  T0: 2→1  earliest=5 
  T1: 3→2→1→0  earliest=2 
  T2: 0→1→2→3→4  earliest=0 
  T3: 2→1→0  earliest=5 
  T4: 2→1→0  earliest=3 ★3.8
  T5: 0→1→2→3  earliest=0 


In [6]:
# Train details
for t in inst.trains:
    path = inst.graph.path(t.origin, t.destination)
    print(f"Train {t.id}:")
    print(f"  Route: {' → '.join(map(str, path))}")
    print(f"  Earliest departure: {t.earliest}")
    print(f"  Priority: {t.priority}")
    print()

Train 0:
  Route: 2 → 1
  Earliest departure: 5
  Priority: 1.0

Train 1:
  Route: 3 → 2 → 1 → 0
  Earliest departure: 2
  Priority: 1.0

Train 2:
  Route: 0 → 1 → 2 → 3 → 4
  Earliest departure: 0
  Priority: 1.0

Train 3:
  Route: 2 → 1 → 0
  Earliest departure: 5
  Priority: 1.0

Train 4:
  Route: 2 → 1 → 0
  Earliest departure: 3
  Priority: 3.8

Train 5:
  Route: 0 → 1 → 2 → 3
  Earliest departure: 0
  Priority: 1.0



---
## 4. Solution Format

A solution is simply: **when does each train depart?**

```python
schedule = {
    0: 5,   # Train 0 departs at t=5
    1: 8,   # Train 1 departs at t=8
    ...
}
```

In [7]:
# Naive solution: everyone departs at earliest time
naive_schedule = naive(inst)
print("Naive schedule:")
print(naive_schedule)

Naive schedule:
{0: 5, 1: 2, 2: 0, 3: 5, 4: 3, 5: 0}


---
## 5. Fitness Evaluation

```
fitness = delay + conflicts + makespan
          (lower = better)
```

| Component | Weight | Description |
|-----------|--------|-------------|
| Delay | 1.0 | `(departure - earliest) × priority` |
| Conflicts | 50.0 | Trains on same segment at same time |
| Makespan | 0.1 | Total schedule duration |

In [8]:
# Evaluate naive solution
result = evaluate(inst, naive_schedule)

print(f"Total Score: {result.score} (lower = better)")
print(f"  Delay:     {result.delay}")
print(f"  Conflicts: {result.conflicts} ({result.num_conflicts} conflicts)")
print(f"  Makespan:  {result.makespan}")

Total Score: 701.1 (lower = better)
  Delay:     0.0
  Conflicts: 700.0 (14 conflicts)
  Makespan:  1.1


---
## 6. Visualization

In [9]:
# Schedule timeline
print(show_schedule(inst, naive_schedule))

Schedule Timeline:

  t=  0: T2 0→1→2→3→4 [arr=11] 
  t=  0: T5 0→1→2→3 [arr=8] 
  t=  2: T1 3→2→1→0 [arr=10] 
  t=  3: T4 2→1→0 [arr=7] 
  t=  5: T0 2→1 [arr=7] 
  t=  5: T3 2→1→0 [arr=9] 

⚠ Conflicts: 14
  same_dir on (2, 1) at t=5: trains [0, 3]
  same_dir on (2, 1) at t=6: trains [0, 1, 3]
  head_on on (3, 2) at t=4: trains [1, 2, 5]
  head_on on (3, 2) at t=5: trains [1, 2, 5]
  same_dir on (1, 0) at t=8: trains [1, 3]
  ... and 9 more


In [10]:
# Time-space diagram
# Rows = stations, Columns = time
# X = conflict!
print(show_timespace(inst, naive_schedule))

Time-Space Diagram (station × time)
Scale: 1 char = 1 time units

     0         10        20        30        40        50        
S0   X····44XXX1·················································
S1   XXX44XXX1···················································
S2   ··XXXX1·····················································
S3   ··1·XXXXX···················································
S4   ········2222················································


---
## 7. Compare Baseline Solvers

In [11]:
results = benchmark(inst)
print_benchmark(results)

Solver                    Score  Conflicts      Delay
----------------------------------------------------
direction_wave            309.3          5       57.4
naive                     701.1         14        0.0
greedy_fcfs               701.1         14        0.0
greedy_priority           701.1         14        0.0


In [12]:
# Best solution visualization
best_name = min(results, key=lambda k: results[k].score)
best_solver = {"naive": naive, "greedy_fcfs": greedy_fcfs, 
               "greedy_priority": greedy_priority, "direction_wave": direction_wave}[best_name]

print(f"Best: {best_name}\n")
print(show_timespace(inst, best_solver(inst)))

Best: direction_wave

Time-Space Diagram (station × time)
Scale: 1 char = 1 time units

     0         10        20        30        40        50        
S0   2·5··········44X1X33········································
S1   22X55······44XXX33··········································
S2   ··22X55··11X1X·3············································
S3   ····22XXXX5·················································
S4   ········2222················································


---
## 8. Harder Instance

In [13]:
# More trains = more conflicts
hard_inst = generate(num_stations=7, num_trains=12, seed=123)

print(show_graph(hard_inst.graph))
print()
print(show_trains(hard_inst))

Graph: |V|=7, |E|=6

[0]--2--[1]--4--[2]--2--[3]--5--[4]--4--[5]--2--[6]

Trains: 12

  T0: 2→1→0  earliest=1 
  T1: 4→3→2  earliest=0 
  T2: 4→3→2  earliest=5 
  T3: 3→4→5→6  earliest=11 
  T4: 5→4→3  earliest=0 
  T5: 1→2→3→4→5→6  earliest=9 ★3.7
  T6: 2→3  earliest=24 
  T7: 4→5→6  earliest=0 
  T8: 6→5  earliest=15 
  T9: 2→3→4  earliest=6 ★4.0
  T10: 1→2  earliest=11 
  T11: 3→4  earliest=0 


In [14]:
print("Baseline comparison on harder instance:\n")
hard_results = benchmark(hard_inst)
print_benchmark(hard_results)

Baseline comparison on harder instance:

Solver                    Score  Conflicts      Delay
----------------------------------------------------
direction_wave            562.4          8      158.0
naive                    1102.6         22        0.0
greedy_fcfs              1102.6         22        0.0
greedy_priority          1102.6         22        0.0


---
## 9. Branching Network

In [15]:
# Non-linear topology
branch_inst = generate(num_stations=8, num_trains=10, topology="branching", seed=42)

print(show_graph(branch_inst.graph))
print()
print(show_trains(branch_inst))

Graph: |V|=8, |E|=4

Adjacency:
  0 -> 1(2)
  1 -> 0(2), 2(2), 4(2)
  2 -> 1(2), 3(4)
  3 -> 2(4)
  4 -> 1(2)
  5 -> 
  6 -> 
  7 -> 

Trains: 10

  T0: 2→1  earliest=6 
  T1: 1→2→3  earliest=6 
  T2: 0→1→2  earliest=0 
  T3: 1→4  earliest=19 ★3.1
  T4: 3→2→1  earliest=3 ★4.6
  T5: 3→2→1→4  earliest=21 
  T6: 2→3  earliest=0 ★3.9
  T7: 0→1→2→3  earliest=25 
  T8: 4→1  earliest=8 
  T9: 4→1→2  earliest=10 


In [16]:
print(branch_inst.graph.to_dot())

graph Rail {
  rankdir=LR;
  node [shape=circle, style=filled, fillcolor=lightblue];
  0 -- 1 [label="2"];
  1 -- 2 [label="2"];
  2 -- 3 [label="4"];
  1 -- 4 [label="2"];
}


---
## 10. ShinkaEvolve Integration

The `DTDPProblem` class provides the interface for algorithm evolution.

In [17]:
# Create problem with difficulty level
problem = DTDPProblem(difficulty="medium")

print(f"Test instances: {len(problem.instances)}")
for i, inst in enumerate(problem.instances):
    print(f"  [{i}] {inst.num_stations} stations, {inst.num_trains} trains")

Test instances: 3
  [0] 5 stations, 8 trains
  [1] 6 stations, 10 trains
  [2] 7 stations, 10 trains


In [18]:
# Get solver template
print(problem.get_template())


def solve(inst: Instance) -> Schedule:
    """
    Solve the DTDP instance.
    
    Args:
        inst: Problem instance with:
            - inst.graph: RailGraph with vertices, edges
            - inst.trains: List[Train] with id, origin, destination, earliest, priority
            - inst.horizon: Maximum time
    
    Returns:
        Schedule: Dict[train_id, departure_time]
    
    Available helpers:
        - inst.graph.path(origin, dest) -> List[int] or None
        - inst.graph.travel_time(u, v) -> int
        - inst.graph.neighbors(v) -> List[int]
    """
    schedule = {}
    
    # YOUR ALGORITHM HERE
    # Example: schedule each train at its earliest time
    for train in inst.trains:
        schedule[train.id] = train.earliest
    
    return schedule



In [ ]:
# Evaluate baseline
score = problem.evaluate(direction_wave)
print(f"Direction wave score: {score}")

In [ ]:
# Get feedback (for LLM learning)
print(problem.get_feedback())

---
## 11. Write Your Own Solver

Try to beat the baselines!

In [ ]:
def my_solver(inst: Instance) -> Schedule:
    """
    Your algorithm here!
    
    Available:
        inst.graph.path(origin, dest) -> [stations] or None
        inst.graph.travel_time(u, v) -> int
        inst.graph.neighbors(v) -> [stations]
        inst.trains -> List[Train]
        inst.horizon -> int
    
    Train attributes:
        train.id, train.origin, train.destination
        train.earliest, train.priority
    """
    schedule = {}
    
    # Example: sort by priority, add spacing
    sorted_trains = sorted(inst.trains, key=lambda t: (-t.priority, t.earliest))
    
    time_cursor = 0
    for train in sorted_trains:
        schedule[train.id] = max(train.earliest, time_cursor)
        time_cursor = schedule[train.id] + 3
    
    return schedule

# Test it
score = problem.evaluate(my_solver)
print(f"My solver score: {score}")
print()
print(problem.get_feedback())

---
## 12. Full Benchmark

In [ ]:
# Test all solvers on full test suite
full_problem = DTDPProblem(difficulty="full")

print("Full benchmark (8 instances):\n")
for name, solver in [("naive", naive), ("greedy_fcfs", greedy_fcfs), 
                      ("greedy_priority", greedy_priority), 
                      ("direction_wave", direction_wave),
                      ("my_solver", my_solver)]:
    score = full_problem.evaluate(solver)
    print(f"{name:<20} {score:>10.1f}")

---
## Summary

**Problem:** Schedule trains to minimize conflicts on single-track railways

**Graph:** G = (V, E) where V = stations, E = track segments

**Solution:** `Dict[train_id, departure_time]`

**Fitness:** `delay + conflicts×50 + makespan×0.1` (lower = better)

**ShinkaEvolve:**
```python
problem = DTDPProblem(difficulty="medium")
score = problem.evaluate(my_solver)
feedback = problem.get_feedback()
```

**Goal:** Beat `direction_wave` baseline!